In [ ]:
from huggingface_hub import login

login(token="")

In [2]:
from datasets import load_dataset, DatasetDict, Audio

common_voice = DatasetDict()
common_voice["train"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="train+validation", trust_remote_code=True)
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="test", trust_remote_code=True)


Using the latest cached version of the module from C:\Users\chapp\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Mon Oct 13 11:51:01 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\chapp\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Mon Oct 13 11:51:01 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.


In [3]:
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [4]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="amharic", task="transcribe")

In [ ]:
import torch
from tqdm import tqdm
import evaluate

wer_metric = evaluate.load("wer")

predictions, references = [], []
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

for example in tqdm(common_voice["test"]):
    input_features = processor(
        example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    predictions.append(transcription)
    references.append(example["sentence"])

wer = 100 * wer_metric.compute(predictions=predictions, references=references)
print(f"Baseline WER (pretrained model): {wer:.2f}%")

100%|██████████| 205/205 [05:29<00:00,  1.61s/it]

Baseline WER (pretrained model): 231.58%
